In [ ]:
from google.colab import files
uploaded = files.upload()     # Select your STL file
stl_filename = list(uploaded.keys())[0]
print("Using:", stl_filename)


Saving liver_lobule.stl to liver_lobule.stl
Using: liver_lobule.stl


In [ ]:
# If uploaded
stl_filename = list(uploaded.keys())[0]


In [ ]:
import numpy as np
from stl import mesh
import matplotlib.pyplot as plt
from scipy.spatial import KDTree, ConvexHull, distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
from matplotlib.path import Path
import collections

# --- STL and geometry ---
stl_filename = "liver_lobule.stl"
your_mesh = mesh.Mesh.from_file(stl_filename)
all_points = your_mesh.vectors.reshape(-1, 3)
z_top = np.max(all_points[:, 2])
z_bottom = np.min(all_points[:, 2])
top_points = all_points[all_points[:, 2] >
                        np.percentile(all_points[:, 2], 95)-1e-2]
xy = top_points[:, :2]
centroid = xy.mean(axis=0)
hull = ConvexHull(xy)
hex_boundary = xy[hull.vertices]
polygon = Path(hex_boundary)

# Inlets: portal triads
angles = np.arctan2(xy[:, 1] - centroid[1], xy[:, 0] - centroid[0])
unique_angles = np.linspace(-np.pi, np.pi, 7)[:-1]
inlet_pts = np.vstack([top_points[np.argmin(np.abs(angles-a))]
                      for a in unique_angles])

# Outlets: central veins (base)
n_ports = 6
outlet_radius = (np.ptp(all_points[:, 0])+np.ptp(all_points[:, 1]))*0.06
outlet_angles = np.linspace(0, 2*np.pi, n_ports, endpoint=False)
outlet_nodes = np.array([
    [
        centroid[0] + outlet_radius * np.cos(a),
        centroid[1] + outlet_radius * np.sin(a),
        z_bottom
    ] for a in outlet_angles
])

# Attraction/perfusion points


def scatter_points(n_points):
    min_x, min_y = np.min(hex_boundary, axis=0)
    max_x, max_y = np.max(hex_boundary, axis=0)
    points = []
    while len(points) < n_points:
        x = np.random.uniform(min_x, max_x)
        y = np.random.uniform(min_y, max_y)
        if not polygon.contains_point([x, y]):
            continue
        z = np.random.uniform(z_bottom, z_top)
        points.append([x, y, z])
    return np.array(points)


n_attraction = 500
attraction_points = scatter_points(n_attraction)

# Assign points to inlets/outlets
kdt_inlets = KDTree(inlet_pts)
arterial_group = kdt_inlets.query(attraction_points)[1]
kdt_outlets = KDTree(outlet_nodes)
venous_group = kdt_outlets.query(attraction_points)[1]

# --- Build arterial and venous trees as MST for each group ---
ID = 0
node_xyz = {}
all_arterial_segs = []
all_venous_segs = []
tree = collections.defaultdict(list)
diameters = {}

# Arterial trees
for inlet_idx, inlet_xyz in enumerate(inlet_pts):
    group_pts = attraction_points[arterial_group == inlet_idx]
    nodes = np.vstack([inlet_xyz, group_pts])
    node_ids = list(range(ID, ID+len(nodes)))
    for nid, xyz in zip(node_ids, nodes):
        node_xyz[nid] = xyz
    ID += len(nodes)
    D = distance_matrix(nodes, nodes)
    mst = minimum_spanning_tree(D)
    mst = mst.toarray().astype(float)
    sources, targets = np.where(mst)
    for s, t in zip(sources, targets):
        a, b = node_ids[s], node_ids[t]
        all_arterial_segs.append((a, b))
        tree[a].append(b)
        diameters[a] = None
    diameters[node_ids[0]] = 0.14  # Set root diameter

# Venous trees
for outlet_idx, outlet_xyz in enumerate(outlet_nodes):
    group_pts = attraction_points[venous_group == outlet_idx]
    if len(group_pts) == 0:
        continue
    nodes = np.vstack([outlet_xyz, group_pts])
    node_ids = list(range(ID, ID+len(nodes)))
    for nid, xyz in zip(node_ids, nodes):
        node_xyz[nid] = xyz
    ID += len(nodes)
    D = distance_matrix(nodes, nodes)
    mst = minimum_spanning_tree(D)
    mst = mst.toarray().astype(float)
    sources, targets = np.where(mst)
    for s, t in zip(sources, targets):
        a, b = node_ids[s], node_ids[t]
        all_venous_segs.append((a, b))
        tree[a].append(b)
        diameters[a] = None
    diameters[node_ids[0]] = 0.13  # Root venous diameter

# Murray's Law (for both trees)


def assign_diameters(root, min_d=0.025):
    if not tree[root]:
        diameters[root] = min_d
        return diameters[root]
    child_ds = []
    for child in tree[root]:
        d = assign_diameters(child, min_d)
        child_ds.append(d)
    parent_d = (sum([d*3 for d in child_ds]))*(1/3)
    diameters[root] = parent_d
    return parent_d


for idx in range(len(inlet_pts)):
    root = None
    for k, v in node_xyz.items():
        if np.allclose(v, inlet_pts[idx]):
            root = k
            break
    if root is not None:
        assign_diameters(root)
for idx in range(len(outlet_nodes)):
    root = None
    for k, v in node_xyz.items():
        if np.allclose(v, outlet_nodes[idx]):
            root = k
            break
    if root is not None:
        assign_diameters(root)

# --- Plotting ---
fig = plt.figure(figsize=(13, 9))
ax = fig.add_subplot(111, projection='3d')
poly_col = Poly3DCollection(
    your_mesh.vectors,
    alpha=0.09,
    facecolor='#eec2da',
    edgecolor='gray',
    linewidths=0.14)
ax.add_collection3d(poly_col)
ax.add_collection3d(Line3DCollection(your_mesh.vectors,
                    colors='gray', linewidths=0.07, alpha=0.10))
ax.scatter(inlet_pts[:, 0], inlet_pts[:, 1], inlet_pts[:,
           2], c='red', s=130, label='Arterial Inlets')
ax.scatter(outlet_nodes[:, 0], outlet_nodes[:, 1],
           outlet_nodes[:, 2], c='blue', s=115, label='Venous Outlets')
ax.scatter(attraction_points[:, 0], attraction_points[:, 1],
           attraction_points[:, 2], c='orange', s=11, alpha=0.48, label='Perfusion Points')

for a, b in all_arterial_segs:
    start, end = node_xyz[a], node_xyz[b]
    diam = diameters.get(a, 0.025)
    if diam is None:
        diam = 0.025
    lw = 1.2 + 5*(diam/0.14)**2
    ax.plot([start[0], end[0]], [start[1], end[1]], [
            start[2], end[2]], color='crimson', linewidth=lw, alpha=0.99)
for a, b in all_venous_segs:
    start, end = node_xyz[a], node_xyz[b]
    diam = diameters.get(a, 0.025)
    if diam is None:
        diam = 0.025
    lw = 1.2 + 5*(diam/0.13)**2
    ax.plot([start[0], end[0]], [start[1], end[1]], [
            start[2], end[2]], color='navy', linewidth=lw, alpha=0.99)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_box_aspect([np.ptp(all_points[:, 0]), np.ptp(
    all_points[:, 1]), np.ptp(all_points[:, 2])])
ax.set_xlim(np.min(all_points[:, 0]), np.max(all_points[:, 0]))
ax.set_ylim(np.min(all_points[:, 1]), np.max(all_points[:, 1]))
ax.set_zlim(np.min(all_points[:, 2]), np.max(all_points[:, 2]))
ax.view_init(elev=25, azim=35)
plt.title("3D Liver Lobule Vasculature",
          fontsize=15, weight='bold')
plt.legend(loc='upper left', markerscale=1.2)
plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'liver_lobule.stl'

In [ ]:
!pip install --quiet plotly


In [ ]:
# Single-cell: 6 unique inlets (one per hex vertex), central vein, O2 colorbar, visible capillaries
# Run in Colab/Jupyter after uploading/mounting liver_lobule.stl
import numpy as np
from stl import mesh
from scipy.spatial import cKDTree, ConvexHull, distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve
from matplotlib.path import Path
import plotly.graph_objects as go
import collections

# ---------- PARAMETERS ----------
stl_filename = "liver_lobule.stl"
n_attraction = 700
knn = 8
min_d = 0.02     # terminal diameter (mm)
root_art_d = 0.14
root_ven_d = 0.13

# ---------- LOAD MESH & HEX ----------
your_mesh = mesh.Mesh.from_file(stl_filename)
all_points = your_mesh.vectors.reshape(-1, 3)
z_top = np.max(all_points[:, 2])
z_bottom = np.min(all_points[:, 2])

top_points = all_points[all_points[:, 2] > np.percentile(all_points[:, 2], 95) - 1e-2]
xy = top_points[:, :2]
centroid = xy.mean(axis=0)
hull = ConvexHull(xy)
hex_boundary = xy[hull.vertices]
polygon = Path(hex_boundary)

# ---------- robust: exactly 6 inlet positions (one per angular bin) ----------
# compute angle and radial distance for each top_point
vecs = xy - centroid
angles = np.arctan2(vecs[:,1], vecs[:,0])  # -pi..pi
r = np.linalg.norm(vecs, axis=1)

# define 6 angular bins centered evenly
bin_edges = np.linspace(-np.pi, np.pi, 7)  # 6 bins
bin_idx = np.digitize(angles, bin_edges) - 1
bin_idx = np.mod(bin_idx, 6)  # ensure 0..5

inlet_pts = []
for b in range(6):
    inds = np.where(bin_idx == b)[0]
    if inds.size == 0:
        # fallback: choose centroid-projected point along bin center
        center_angle = 0.5*(bin_edges[b] + bin_edges[b+1])
        # pick top_point with minimal angular distance to center
        ang_diff = np.abs(np.angle(np.exp(1j*(angles - center_angle))))
        chosen = np.argmin(ang_diff)
    else:
        # choose the farthest radial point in the bin (gives an edge vertex)
        chosen_local = inds[np.argmax(r[inds])]
        chosen = chosen_local
    inlet_pts.append(top_points[chosen])
inlet_pts = np.array(inlet_pts)  # shape (6,3)

# ensure uniqueness (should be unique by bin method)
# but if two bins picked the same index (rare), perturb by selecting next-best
# create set of tuples; if duplicate, replace by nearest unused top_point in that bin
used = set()
for i in range(len(inlet_pts)):
    tup = tuple(np.round(inlet_pts[i], 8))
    if tup in used:
        # find another in same bin
        b = i
        inds = np.where(bin_idx == b)[0]
        # choose next farthest not used
        sorted_inds = inds[np.argsort(-r[inds])]
        replacement = None
        for si in sorted_inds:
            cand = tuple(np.round(top_points[si], 8))
            if cand not in used:
                replacement = top_points[si]; break
        if replacement is not None:
            inlet_pts[i] = replacement
            used.add(tuple(np.round(replacement, 8)))
    else:
        used.add(tup)

# central vein at centroid (single outlet)
central_vein = np.array([centroid[0], centroid[1], z_bottom]).reshape(1,3)

# ---------- attraction points inside hex ----------
def scatter_points(n):
    min_x, min_y = np.min(hex_boundary, axis=0)
    max_x, max_y = np.max(hex_boundary, axis=0)
    pts=[]
    while len(pts) < n:
        x = np.random.uniform(min_x, max_x); y = np.random.uniform(min_y, max_y)
        if not polygon.contains_point([x,y]): continue
        z = np.random.uniform(z_bottom, z_top)
        pts.append([x,y,z])
    return np.array(pts)

attraction_points = scatter_points(n_attraction)

# ---------- Build k-NN Laplacian on full node set (for O2) ----------
# nodes for Laplacian: inlet_pts (6), central_vein (1), attraction_points
nodes = np.vstack([inlet_pts, central_vein, attraction_points])
n_inlets = len(inlet_pts)
idx_inlets = np.arange(0, n_inlets)
idx_central = n_inlets
idx_attract = np.arange(n_inlets+1, nodes.shape[0])

# build symmetric weighted adjacency using k-NN
kdt = cKDTree(nodes)
dists, neigh = kdt.query(nodes, k=knn+1)  # includes self
rows = []; cols = []; data = []
for i in range(nodes.shape[0]):
    for j in neigh[i,1:]:
        dij = np.linalg.norm(nodes[i]-nodes[j])
        if dij == 0: w = 0.0
        else: w = 1.0/(dij + 1e-12)
        rows.append(i); cols.append(j); data.append(w)
# symmetrize
rows_sym = rows + cols; cols_sym = cols + rows; data_sym = data + data
A = csr_matrix((data_sym, (rows_sym, cols_sym)), shape=(nodes.shape[0], nodes.shape[0]))
deg = np.array(A.sum(axis=1)).flatten()
D = csr_matrix((deg, (np.arange(nodes.shape[0]), np.arange(nodes.shape[0]))), shape=(nodes.shape[0], nodes.shape[0]))
L = D - A

# Solve discrete oxygen field (Dirichlet): inlets -> 1.0 ; central vein -> 0.0
boundary_idx = np.concatenate([idx_inlets, [idx_central]])
interior_idx = np.setdiff1d(np.arange(nodes.shape[0]), boundary_idx)

L_uu = L[ np.ix_(interior_idx, interior_idx) ]
L_ub = L[ np.ix_(interior_idx, boundary_idx) ]
o_b = np.concatenate([np.ones(len(idx_inlets)), np.zeros(1)])  # inlets O2=1, central=0
o_u = spsolve(L_uu, - L_ub.dot(o_b))
O = np.zeros(nodes.shape[0])
O[boundary_idx] = o_b
O[interior_idx] = o_u

# attraction O2 values
o_attract = O[idx_attract]

# ---------- Exclusive assignment: each attraction point -> nearest of (6 inlets, central) ----------
k_roots = cKDTree(np.vstack([inlet_pts, central_vein]))
droot, iroot = k_roots.query(attraction_points)
assigned_to_inlet = (iroot < n_inlets)
arterial_points = attraction_points[assigned_to_inlet]
venous_points   = attraction_points[~assigned_to_inlet]
# map which inlet each arterial point belongs to (0..5)
arterial_root_idx_for_point = iroot[assigned_to_inlet]  # indices into inlet_pts

# ---------- Build arterial MSTs (one per inlet) ----------
node_xyz = {}
ID = 0
all_arterial_segs = []
all_venous_segs = []
tree = collections.defaultdict(list)
diameters = {}

# For each inlet build MST on (inlet + its assigned attraction points)
for inlet_idx in range(n_inlets):
    mask = (arterial_root_idx_for_point == inlet_idx)
    group_pts = arterial_points[mask] if len(arterial_points)>0 else np.empty((0,3))
    nodes_art = np.vstack([inlet_pts[inlet_idx].reshape(1,3), group_pts]) if len(group_pts)>0 else np.vstack([inlet_pts[inlet_idx].reshape(1,3)])
    ids = list(range(ID, ID+len(nodes_art)))
    for nid, xyz in zip(ids, nodes_art):
        node_xyz[nid] = xyz
    ID += len(nodes_art)
    D = distance_matrix(nodes_art, nodes_art)
    mst = minimum_spanning_tree(D).toarray().astype(float)
    s,t = np.where(mst)
    for si,ti in zip(s,t):
        a = ids[si]; b = ids[ti]
        all_arterial_segs.append((a,b))
        tree[a].append(b)
        diameters[a] = None
    diameters[ids[0]] = root_art_d

# ---------- Build venous MST toward central vein ----------
ven_nodes = np.vstack([central_vein.reshape(1,3), venous_points]) if len(venous_points)>0 else np.vstack([central_vein.reshape(1,3)])
ven_ids = list(range(ID, ID+len(ven_nodes)))
for nid, xyz in zip(ven_ids, ven_nodes):
    node_xyz[nid] = xyz
ID += len(ven_nodes)
Dv = distance_matrix(ven_nodes, ven_nodes)
mst_v = minimum_spanning_tree(Dv).toarray().astype(float)
s,t = np.where(mst_v)
for si,ti in zip(s,t):
    a = ven_ids[si]; b = ven_ids[ti]
    all_venous_segs.append((a,b))
    tree[a].append(b)
    diameters[a] = None
diameters[ven_ids[0]] = root_ven_d

# ---------- Capillaries: arterial leaves -> nearest venous leaves (highlight thicker & bright) ----------
def leaves(node_set):
    return [n for n in node_set if len(tree.get(n, [])) == 0]

arterial_node_set = set([a for seg in all_arterial_segs for a in seg])
# ensure inlet roots included
arterial_node_set.update([nid for nid,v in node_xyz.items() if any(np.allclose(v, ip) for ip in inlet_pts)])
venous_node_set = set([a for seg in all_venous_segs for a in seg])
venous_node_set.update(ven_ids[:1])

arterial_leaves = leaves(arterial_node_set)
venous_leaves = leaves(venous_node_set)
if len(venous_leaves) == 0:
    venous_leaves = list(venous_node_set)

vleaf_coords = np.array([node_xyz[n] for n in venous_leaves])
k_vleaf = cKDTree(vleaf_coords) if len(vleaf_coords)>0 else None

capillary_segs = []
for a in arterial_leaves:
    if k_vleaf is None: break
    a_coord = node_xyz[a]
    d, idx = k_vleaf.query(a_coord)
    v_global = venous_leaves[idx]
    if np.linalg.norm(node_xyz[a] - node_xyz[v_global]) < 1e-9: continue
    capillary_segs.append((a, v_global))
    tree[a].append(v_global)  # include in tree for Murray
    diameters[a] = diameters.get(a, None)

# ---------- Murray's law (corrected) ----------
def assign_diameters(root, min_d=min_d):
    if not tree.get(root):
        diameters[root] = min_d
        return diameters[root]
    child_ds = []
    for c in tree[root]:
        d = assign_diameters(c, min_d)
        child_ds.append(d)
    parent_d = (sum([d**3 for d in child_ds]))**(1/3)
    diameters[root] = parent_d
    return parent_d

# assign for all inlet roots
for inlet_xyz in inlet_pts:
    root = next((k for k,v in node_xyz.items() if np.allclose(v, inlet_xyz)), None)
    if root is not None:
        assign_diameters(root)
# assign for central vein
ven_root = next((k for k,v in node_xyz.items() if np.allclose(v, central_vein.reshape(3))), None)
if ven_root is not None:
    assign_diameters(ven_root)

# ---------- mapping diam->linewidth ----------
def diam_to_lw(d, dmin=min_d, dmax=root_art_d):
    t = (d - dmin) / max(1e-9, (dmax - dmin))
    return 0.6 + 7.4 * np.clip(t, 0.0, 1.0)

# build per-segment Plotly traces (one per seg to allow custom widths/colors)
def seg_traces(segs, color, name, width_scale=1.0, dash=None):
    traces=[]
    for idx, (a,b) in enumerate(segs):
        s=node_xyz[a]; e=node_xyz[b]
        da = diameters.get(a, min_d) or min_d
        db = diameters.get(b, min_d) or min_d
        lw = diam_to_lw((da+db)/2.0) * width_scale
        trace = go.Scatter3d(x=[s[0], e[0]], y=[s[1], e[1]], z=[s[2], e[2]],
                             mode='lines',
                             line=dict(color=color, width=lw, dash=dash if dash else 'solid'),
                             showlegend=(idx==0), name=name if idx==0 else None)
        traces.append(trace)
    return traces

# ---------- prepare mesh ----------
verts, inv = np.unique(your_mesh.vectors.reshape(-1,3), axis=0, return_inverse=True)
faces = inv.reshape(-1,3)
i = faces[:,0]; j = faces[:,1]; k = faces[:,2]

fig = go.Figure()
fig.add_trace(go.Mesh3d(x=verts[:,0], y=verts[:,1], z=verts[:,2],
                       i=i,j=j,k=k, opacity=0.20, flatshading=True, name='Lobule surface'))

# arterial (all 6)
for tr in seg_traces(all_arterial_segs, 'crimson', 'Arterial (inlets)', width_scale=1.0):
    fig.add_trace(tr)

# venous (central)
for tr in seg_traces(all_venous_segs, 'navy', 'Venous (central)', width_scale=1.0):
    fig.add_trace(tr)

# capillaries: make visibly distinct (bright yellow and thicker)
for tr in seg_traces(capillary_segs, 'gold', 'Capillaries', width_scale=1.6, dash='solid'):
    fig.add_trace(tr)

# markers: inlet pts (6) and central vein
fig.add_trace(go.Scatter3d(x=inlet_pts[:,0], y=inlet_pts[:,1], z=inlet_pts[:,2],
                           mode='markers', marker=dict(size=6, symbol='diamond', color='red'), name='Inlets (6)'))
fig.add_trace(go.Scatter3d(x=[central_vein[0,0]], y=[central_vein[0,1]], z=[central_vein[0,2]],
                           mode='markers', marker=dict(size=7, symbol='circle', color='blue'), name='Central vein'))

# attraction pts colored by O2 with colorbar
fig.add_trace(go.Scatter3d(x=attraction_points[:,0], y=attraction_points[:,1], z=attraction_points[:,2],
                           mode='markers',
                           marker=dict(size=3, color=o_attract, colorscale='Viridis', colorbar=dict(title='O₂'), showscale=True),
                           name='Attraction pts (O2)'))

fig.update_layout(scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
                  width=1000, height=820)
fig.show()

# ---------- diagnostics ----------
print("Inlets (6) coords — one per vertex:")
for i,c in enumerate(inlet_pts):
    print(i, c)
print("Central vein:", central_vein.reshape(3))
print("Arterial segs:", len(all_arterial_segs))
print("Venous segs:", len(all_venous_segs))
print("Capillary segs:", len(capillary_segs))


Inlets (6) coords — one per vertex:
0 [-9.742786 -5.625     6.1     ]
1 [-1.76043e-15 -1.12500e+01  6.10000e+00]
2 [ 9.742786 -5.625     6.1     ]
3 [9.742786 5.625    6.1     ]
4 [3.827021e-16 1.125000e+01 6.100000e+00]
5 [-9.742786  5.625     6.1     ]
Central vein: [-6.328394e-08 -8.506539e-07 -6.000000e+00]
Arterial segs: 411
Venous segs: 289
Capillary segs: 152


In [ ]:
# === Add VEGF field + produce 3 static pictures + interactive 3D vessel plot ===
# Run in same Colab session after you already have your network variables,
# or run standalone (it rebuilds the network with sensible defaults).
# If you already have node_xyz, all_arterial_segs, all_venous_segs, capillary_segment_edges,
# attraction_points, o_attract, inlet_pts, central_vein, diameters in memory, it'll reuse them;
# otherwise it builds them (light defaults).

import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import plotly.graph_objects as go
from stl import mesh
from scipy.spatial import cKDTree, ConvexHull
from scipy.spatial import distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from matplotlib.path import Path
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve
import collections

# ---------- PARAMETERS (keep consistent with your previous run) ----------
stl_filename = "liver_lobule.stl"
n_attraction = 900     # increase if you want finer resolution
knn = 8
min_d = 0.02
root_art_d = 0.14
root_ven_d = 0.13

# ---------- helper: build network if not present ----------
need_build = ('node_xyz' not in globals()) or ('all_arterial_segs' not in globals())

if need_build:
    print("Building network (this can take ~30-60s)...")
    your_mesh = mesh.Mesh.from_file(stl_filename)
    all_points = your_mesh.vectors.reshape(-1, 3)
    z_top = np.max(all_points[:, 2]); z_bottom = np.min(all_points[:, 2])
    top_points = all_points[all_points[:, 2] > np.percentile(all_points[:, 2], 95) - 1e-2]
    xy = top_points[:, :2]
    centroid = xy.mean(axis=0)
    hull = ConvexHull(xy)
    hex_boundary = xy[hull.vertices]
    polygon = Path(hex_boundary)

    # exactly 6 inlet positions (robust bin method)
    vecs = xy - centroid
    angles = np.arctan2(vecs[:,1], vecs[:,0])
    r = np.linalg.norm(vecs, axis=1)
    bin_edges = np.linspace(-np.pi, np.pi, 7)
    bin_idx = np.digitize(angles, bin_edges) - 1
    bin_idx = np.mod(bin_idx, 6)

    inlet_pts = []
    for b in range(6):
        inds = np.where(bin_idx == b)[0]
        if inds.size == 0:
            center_angle = 0.5*(bin_edges[b] + bin_edges[b+1])
            ang_diff = np.abs(np.angle(np.exp(1j*(angles - center_angle))))
            chosen = np.argmin(ang_diff)
        else:
            chosen_local = inds[np.argmax(r[inds])]
            chosen = chosen_local
        inlet_pts.append(top_points[chosen])
    inlet_pts = np.array(inlet_pts)
    central_vein = np.array([centroid[0], centroid[1], z_bottom]).reshape(1,3)

    # attraction points
    def scatter_points(n):
        min_x, min_y = np.min(hex_boundary, axis=0)
        max_x, max_y = np.max(hex_boundary, axis=0)
        pts=[]
        while len(pts) < n:
            x = np.random.uniform(min_x, max_x); y = np.random.uniform(min_y, max_y)
            if not polygon.contains_point([x,y]): continue
            z = np.random.uniform(z_bottom, z_top)
            pts.append([x,y,z])
        return np.array(pts)

    attraction_points = scatter_points(n_attraction)

    # O2 Laplacian (for color)
    nodes = np.vstack([inlet_pts, central_vein, attraction_points])
    n_inlets = len(inlet_pts)
    idx_inlets = np.arange(n_inlets)
    idx_central = n_inlets
    idx_attract = np.arange(n_inlets+1, nodes.shape[0])

    kdt = cKDTree(nodes)
    dists, neigh = kdt.query(nodes, k=knn+1)
    rows = []; cols = []; data = []
    for i in range(nodes.shape[0]):
        for j in neigh[i,1:]:
            dij = np.linalg.norm(nodes[i]-nodes[j])
            if dij == 0: w = 0.0
            else: w = 1.0/(dij + 1e-12)
            rows.append(i); cols.append(j); data.append(w)
    rows_sym = rows + cols; cols_sym = cols + rows; data_sym = data + data
    A = csr_matrix((data_sym, (rows_sym, cols_sym)), shape=(nodes.shape[0], nodes.shape[0]))
    deg = np.array(A.sum(axis=1)).flatten()
    D = csr_matrix((deg, (np.arange(nodes.shape[0]), np.arange(nodes.shape[0]))), shape=(nodes.shape[0], nodes.shape[0]))
    L = D - A
    boundary_idx = np.concatenate([idx_inlets, [idx_central]])
    interior_idx = np.setdiff1d(np.arange(nodes.shape[0]), boundary_idx)
    L_uu = L[ np.ix_(interior_idx, interior_idx) ]
    L_ub = L[ np.ix_(interior_idx, boundary_idx) ]
    o_b = np.concatenate([np.ones(len(idx_inlets)), np.zeros(1)])
    o_u = spsolve(L_uu, - L_ub.dot(o_b))
    O = np.zeros(nodes.shape[0])
    O[boundary_idx] = o_b
    O[interior_idx] = o_u
    o_attract = O[idx_attract]

    # Exclusive assignment nearest root
    k_roots = cKDTree(np.vstack([inlet_pts, central_vein]))
    droot, iroot = k_roots.query(attraction_points)
    assigned_to_inlet = (iroot < n_inlets)
    arterial_points = attraction_points[assigned_to_inlet]
    venous_points   = attraction_points[~assigned_to_inlet]
    arterial_root_idx_for_point = iroot[assigned_to_inlet]

    # Build arterial MSTs
    node_xyz = {}
    ID = 0
    all_arterial_segs = []
    all_venous_segs = []
    tree = collections.defaultdict(list)
    diameters = {}

    for inlet_idx in range(n_inlets):
        mask = (arterial_root_idx_for_point == inlet_idx)
        group_pts = arterial_points[mask] if len(arterial_points)>0 else np.empty((0,3))
        nodes_art = np.vstack([inlet_pts[inlet_idx].reshape(1,3), group_pts]) if len(group_pts)>0 else np.vstack([inlet_pts[inlet_idx].reshape(1,3)])
        ids = list(range(ID, ID+len(nodes_art)))
        for nid, xyz in zip(ids, nodes_art):
            node_xyz[nid] = xyz
        ID += len(nodes_art)
        D = distance_matrix(nodes_art, nodes_art)
        mst = minimum_spanning_tree(D).toarray().astype(float)
        s,t = np.where(mst)
        for si,ti in zip(s,t):
            a = ids[si]; b = ids[ti]
            all_arterial_segs.append((a,b))
            tree[a].append(b)
            diameters[a] = None
        diameters[ids[0]] = root_art_d

    # Venous MST (central vein)
    ven_nodes = np.vstack([central_vein.reshape(1,3), venous_points]) if len(venous_points)>0 else np.vstack([central_vein.reshape(1,3)])
    ven_ids = list(range(ID, ID+len(ven_nodes)))
    for nid, xyz in zip(ven_ids, ven_nodes):
        node_xyz[nid] = xyz
    ID += len(ven_nodes)
    Dv = distance_matrix(ven_nodes, ven_nodes)
    mst_v = minimum_spanning_tree(Dv).toarray().astype(float)
    s,t = np.where(mst_v)
    for si,ti in zip(s,t):
        a, b = ven_ids[si], ven_ids[ti]
        all_venous_segs.append((a,b))
        tree[a].append(b)
        diameters[a] = None
    diameters[ven_ids[0]] = root_ven_d

    # Simple capillaries: connect arterial leaves -> nearest venous leaves (kept simple)
    def leaves(node_set):
        return [n for n in node_set if len(tree.get(n, [])) == 0]

    arterial_node_set = set([a for seg in all_arterial_segs for a in seg])
    arterial_node_set.update([nid for nid,v in node_xyz.items() if any(np.allclose(v, ip) for ip in inlet_pts)])
    venous_node_set = set([a for seg in all_venous_segs for a in seg])
    venous_node_set.update(ven_ids[:1])

    arterial_leaves = leaves(arterial_node_set)
    venous_leaves = leaves(venous_node_set)
    if len(venous_leaves) == 0:
        venous_leaves = list(venous_node_set)

    v_coords = np.array([node_xyz[n] for n in venous_leaves]) if len(venous_leaves)>0 else np.empty((0,3))
    kv = cKDTree(v_coords) if len(v_coords)>0 else None

    capillary_segs = []
    for a in arterial_leaves:
        if kv is None: break
        d, idx = kv.query(node_xyz[a])
        v_global = venous_leaves[idx]
        if np.linalg.norm(node_xyz[a] - node_xyz[v_global]) < 1e-9:
            continue
        capillary_segs.append((a, v_global))
        tree[a].append(v_global)
        diameters[a] = diameters.get(a, None)

    # Murray's law (same)
    def assign_diameters(root, min_d=min_d):
        if not tree.get(root):
            diameters[root] = min_d
            return diameters[root]
        child_ds = []
        for child in tree[root]:
            d = assign_diameters(child, min_d)
            child_ds.append(d)
        parent_d = (sum([d**3 for d in child_ds]))**(1/3)
        diameters[root] = parent_d
        return parent_d

    for inlet_xyz in inlet_pts:
        root = next((k for k,v in node_xyz.items() if np.allclose(v, inlet_xyz)), None)
        if root is not None:
            assign_diameters(root)
    ven_root = next((k for k,v in node_xyz.items() if np.allclose(v, central_vein.reshape(3))), None)
    if ven_root is not None:
        assign_diameters(ven_root)

    # fill missing diameters
    for nid in node_xyz:
        if diameters.get(nid) is None:
            diameters[nid] = min_d

else:
    print("Using existing network in memory (re-using variables).")
    # ensure required variables exist
    assert 'attraction_points' in globals() and 'o_attract' in globals(), "Run network build first."

# ---------- VEGF field (periportal baseline + hypoxia-induced term) ----------
# compute distance to nearest inlet for each attraction point
k_in = cKDTree(inlet_pts)
dist_to_inlet = k_in.query(attraction_points)[0]                     # min distance to the inlets
max_dist = np.max(dist_to_inlet) if np.max(dist_to_inlet) > 0 else 1.0
base_vegf = 1.0 - (dist_to_inlet / max_dist)                         # 1 near inlet (periportal), 0 far (perivenular)

# hypoxia term from O2 (assuming o_attract in 0..1)
hypoxia_term = 1.0 - o_attract                                       # low O2 -> high hypoxia

# combine
alpha = 0.6   # periportal baseline weight
beta  = 0.4   # hypoxia-induced VEGF weight
vegf_raw = alpha * base_vegf + beta * hypoxia_term

# normalize 0..1
vegf_min, vegf_max = vegf_raw.min(), vegf_raw.max()
if vegf_max - vegf_min < 1e-12:
    vegf = np.clip(vegf_raw, 0, 1)
else:
    vegf = (vegf_raw - vegf_min) / (vegf_max - vegf_min)

# ---------- Color maps ----------
# custom dark->light blue for O2 (we make a 6-color linear map)
blue_colors = [(2/255, 48/255, 71/255), (3/255, 70/255, 120/255), (30/255,100/255,180/255), (120/255,170/255,230/255), (180/255,210/255,250/255)]
cmap_o2 = LinearSegmentedColormap.from_list("dark2lightblue", blue_colors)

# custom dark->light pink for VEGF
pink_colors = [(90/255, 15/255, 45/255), (140/255,35/255,90/255), (200/255,100/255,150/255), (240/255,160/255,200/255), (255/255,220/255,235/255)]
cmap_vegf = LinearSegmentedColormap.from_list("dark2lightpink", pink_colors)

# ---------- Save three static images ----------
os.makedirs("output_images", exist_ok=True)

# 1) oxygen only (attraction points only)
plt.figure(figsize=(8,8))
sc = plt.scatter(attraction_points[:,0], attraction_points[:,1], c=o_attract, cmap=cmap_o2, s=8)
plt.axis('equal'); plt.title("Attraction points colored by O₂")
plt.colorbar(sc, label="O₂ (normalized)")
plt.savefig("output_images/oxygen_only.png", dpi=200, bbox_inches='tight')
plt.close()

# 2) VEGF only
plt.figure(figsize=(8,8))
sc = plt.scatter(attraction_points[:,0], attraction_points[:,1], c=vegf, cmap=cmap_vegf, s=8)
plt.axis('equal'); plt.title("Attraction points colored by VEGF")
plt.colorbar(sc, label="VEGF (normalized)")
plt.savefig("output_images/vegf_only.png", dpi=200, bbox_inches='tight')
plt.close()

# 3) Combined: O2 | VEGF | Overlay
fig, axs = plt.subplots(1,3, figsize=(18,6))
ax = axs[0]
sc0 = ax.scatter(attraction_points[:,0], attraction_points[:,1], c=o_attract, cmap=cmap_o2, s=6)
ax.set_title("O₂")
ax.axis('equal'); plt.colorbar(sc0, ax=ax, fraction=0.046, pad=0.02)

ax = axs[1]
sc1 = ax.scatter(attraction_points[:,0], attraction_points[:,1], c=vegf, cmap=cmap_vegf, s=6)
ax.set_title("VEGF")
ax.axis('equal'); plt.colorbar(sc1, ax=ax, fraction=0.046, pad=0.02)

ax = axs[2]
ax.scatter(attraction_points[:,0], attraction_points[:,1], c=o_attract, cmap=cmap_o2, s=6, label="O2")
ax.scatter(attraction_points[:,0], attraction_points[:,1], c=vegf, cmap=cmap_vegf, s=12, alpha=0.33, label="VEGF")
ax.set_title("Overlay (O₂ + VEGF)")
ax.axis('equal')
axs[2].legend(loc='upper right')

plt.savefig("output_images/combined_oxygen_vegf.png", dpi=200, bbox_inches='tight')
plt.close()

print("Saved images to output_images/: oxygen_only.png, vegf_only.png, combined_oxygen_vegf.png")

# ---------- 3D vessel plot (interactive Plotly) colored by O2 on attraction points,
# capillaries highlighted in gold; optionally overlay VEGF as slightly larger pink markers ----------

# prepare mesh3d vertices (same as earlier)
your_mesh = mesh.Mesh.from_file(stl_filename)
verts, inv = np.unique(your_mesh.vectors.reshape(-1,3), axis=0, return_inverse=True)
faces = inv.reshape(-1,3)
i = faces[:,0]; j = faces[:,1]; k = faces[:,2]

fig = go.Figure()

# mesh
fig.add_trace(go.Mesh3d(
    x=verts[:,0], y=verts[:,1], z=verts[:,2],
    i=i, j=j, k=k,
    opacity=0.22, flatshading=True, name='Lobule surface'
))

# arterial traces
for (a,b) in all_arterial_segs:
    s = node_xyz[a]; e = node_xyz[b]
    da = diameters.get(a, min_d) or min_d
    db = diameters.get(b, min_d) or min_d
    lw = 1.0 + 6.0 * (( (da+db)/2.0 - min_d) / (root_art_d - min_d))
    fig.add_trace(go.Scatter3d(x=[s[0], e[0]], y=[s[1], e[1]], z=[s[2], e[2]],
                               mode='lines', line=dict(color='crimson', width=lw), showlegend=False))

# venous traces
for (a,b) in all_venous_segs:
    s = node_xyz[a]; e = node_xyz[b]
    da = diameters.get(a, min_d) or min_d
    db = diameters.get(b, min_d) or min_d
    lw = 1.0 + 6.0 * (( (da+db)/2.0 - min_d) / (root_art_d - min_d))
    fig.add_trace(go.Scatter3d(x=[s[0], e[0]], y=[s[1], e[1]], z=[s[2], e[2]],
                               mode='lines', line=dict(color='navy', width=lw), showlegend=False))

# capillaries
for (a,b) in capillary_segs:
    s = node_xyz[a]; e = node_xyz[b]
    fig.add_trace(go.Scatter3d(x=[s[0], e[0]], y=[s[1], e[1]], z=[s[2], e[2]],
                               mode='lines', line=dict(color='gold', width=2.2), showlegend=False))

# attraction points colored by O2 (dark->light blue)
fig.add_trace(go.Scatter3d(x=attraction_points[:,0], y=attraction_points[:,1], z=attraction_points[:,2],
                           mode='markers',
                           marker=dict(size=3, color=o_attract, colorscale='Blues', colorbar=dict(title='O₂')),
                           name='Attraction pts (O₂)'))

# overlay VEGF slightly larger transparent pink markers (optional toggle)
fig.add_trace(go.Scatter3d(x=attraction_points[:,0], y=attraction_points[:,1], z=attraction_points[:,2],
                           mode='markers',
                           marker=dict(size=6, color=vegf, colorscale=[[0,'#5a0f2d'],[0.5,'#c86496'],[1,'#ffdceb']], opacity=0.45),
                           name='VEGF (overlay)'))

# inlet and central markers
fig.add_trace(go.Scatter3d(x=inlet_pts[:,0], y=inlet_pts[:,1], z=inlet_pts[:,2],
                           mode='markers', marker=dict(size=6, symbol='diamond', color='red'), name='Inlets'))
fig.add_trace(go.Scatter3d(x=[central_vein[0,0]], y=[central_vein[0,1]], z=[central_vein[0,2]],
                           mode='markers', marker=dict(size=8, symbol='circle', color='blue'), name='Central vein'))

fig.update_layout(scene=dict(aspectmode='data',
               xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
                  width=1000, height=820)
fig.show()

# Save an interactive html snapshot (optional)
import plotly.io as pio
pio.write_html(fig, file="output_images/vessel_3d_plot.html", auto_open=False)
print("Saved interactive 3D plot as output_images/vessel_3d_plot.html")

# Done
print("All visuals generated. Check /output_images for PNGs and HTML.")


Building network (this can take ~30-60s)...


FileNotFoundError: [Errno 2] No such file or directory: 'liver_lobule.stl'

In [ ]:
# === VEGF + Murray fixes + 3 PNGs + 3D vessel plot (capillaries removed) ===
# Paste this cell in Colab / Jupyter and run. It will rebuild network if needed.
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import plotly.graph_objects as go
from stl import mesh
from scipy.spatial import cKDTree, ConvexHull
from scipy.spatial import distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from matplotlib.path import Path
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve
import collections

# ---------- PARAMETERS ----------
stl_filename = "liver_lobule.stl"
n_attraction = 900
knn = 8
min_d = 0.02
root_art_d = 8.0    # user-requested root inlet diameter (mm)
root_ven_d = 8.0
# visualization scaling
def diam_to_lw_visual(d, dmin=min_d, dmax=root_art_d):
    t = (d - dmin) / max(1e-9, (dmax - dmin))
    t = np.clip(t, 0.0, 1.0)
    return 0.8 + 18.0 * (t ** 0.6)

# ---------- Build network if not present ----------
need_build = ('node_xyz' not in globals()) or ('all_arterial_segs' not in globals())
if need_build:
    print("Building network (this can take a short while)...")
    your_mesh = mesh.Mesh.from_file(stl_filename)
    all_points = your_mesh.vectors.reshape(-1, 3)
    z_top = np.max(all_points[:, 2]); z_bottom = np.min(all_points[:, 2])
    top_points = all_points[all_points[:, 2] > np.percentile(all_points[:, 2], 95) - 1e-2]
    xy = top_points[:, :2]
    centroid = xy.mean(axis=0)
    hull = ConvexHull(xy)
    hex_boundary = xy[hull.vertices]
    polygon = Path(hex_boundary)

    # Exactly 6 inlets (angular bins)
    vecs = xy - centroid
    angles = np.arctan2(vecs[:,1], vecs[:,0])
    r = np.linalg.norm(vecs, axis=1)
    bin_edges = np.linspace(-np.pi, np.pi, 7)
    bin_idx = np.digitize(angles, bin_edges) - 1
    bin_idx = np.mod(bin_idx, 6)

    inlet_pts = []
    for b in range(6):
        inds = np.where(bin_idx == b)[0]
        if inds.size == 0:
            center_angle = 0.5*(bin_edges[b] + bin_edges[b+1])
            ang_diff = np.abs(np.angle(np.exp(1j*(angles - center_angle))))
            chosen = np.argmin(ang_diff)
        else:
            chosen_local = inds[np.argmax(r[inds])]
            chosen = chosen_local
        inlet_pts.append(top_points[chosen])
    inlet_pts = np.array(inlet_pts)
    central_vein = np.array([centroid[0], centroid[1], z_bottom]).reshape(1,3)

    # attraction points inside hex
    def scatter_points(n):
        min_x, min_y = np.min(hex_boundary, axis=0)
        max_x, max_y = np.max(hex_boundary, axis=0)
        pts=[]
        while len(pts) < n:
            x = np.random.uniform(min_x, max_x); y = np.random.uniform(min_y, max_y)
            if not polygon.contains_point([x,y]): continue
            z = np.random.uniform(z_bottom, z_top)
            pts.append([x,y,z])
        return np.array(pts)

    attraction_points = scatter_points(n_attraction)

    # Build Laplacian & O2
    nodes = np.vstack([inlet_pts, central_vein, attraction_points])
    n_inlets = len(inlet_pts)
    idx_inlets = np.arange(n_inlets)
    idx_central = n_inlets
    idx_attract = np.arange(n_inlets+1, nodes.shape[0])

    kdt = cKDTree(nodes)
    dists, neigh = kdt.query(nodes, k=knn+1)
    rows=[]; cols=[]; data=[]
    for i in range(nodes.shape[0]):
        for j in neigh[i,1:]:
            dij = np.linalg.norm(nodes[i]-nodes[j])
            if dij == 0: w = 0.0
            else: w = 1.0/(dij + 1e-12)
            rows.append(i); cols.append(j); data.append(w)
    rows_sym = rows + cols; cols_sym = cols + rows; data_sym = data + data
    A = csr_matrix((data_sym, (rows_sym, cols_sym)), shape=(nodes.shape[0], nodes.shape[0]))
    deg = np.array(A.sum(axis=1)).flatten()
    D = csr_matrix((deg, (np.arange(nodes.shape[0]), np.arange(nodes.shape[0]))), shape=(nodes.shape[0], nodes.shape[0]))
    L = D - A
    boundary_idx = np.concatenate([idx_inlets, [idx_central]])
    interior_idx = np.setdiff1d(np.arange(nodes.shape[0]), boundary_idx)
    L_uu = L[np.ix_(interior_idx, interior_idx)]
    L_ub = L[np.ix_(interior_idx, boundary_idx)]
    o_b = np.concatenate([np.ones(len(idx_inlets)), np.zeros(1)])
    o_u = spsolve(L_uu, - L_ub.dot(o_b))
    O = np.zeros(nodes.shape[0])
    O[boundary_idx] = o_b
    O[interior_idx] = o_u
    o_attract = O[idx_attract]

    # Exclusive assign to nearest root (inlets or central)
    k_roots = cKDTree(np.vstack([inlet_pts, central_vein]))
    droot, iroot = k_roots.query(attraction_points)
    assigned_to_inlet = (iroot < n_inlets)
    arterial_points = attraction_points[assigned_to_inlet]
    venous_points   = attraction_points[~assigned_to_inlet]
    arterial_root_idx_for_point = iroot[assigned_to_inlet]

    # Build arterial MSTs (one per inlet)
    node_xyz = {}
    ID = 0
    all_arterial_segs = []
    all_venous_segs = []
    tree = collections.defaultdict(list)
    diameters = {}

    for inlet_idx in range(n_inlets):
        mask = (arterial_root_idx_for_point == inlet_idx)
        group_pts = arterial_points[mask] if len(arterial_points)>0 else np.empty((0,3))
        nodes_art = np.vstack([inlet_pts[inlet_idx].reshape(1,3), group_pts]) if len(group_pts)>0 else np.vstack([inlet_pts[inlet_idx].reshape(1,3)])
        ids = list(range(ID, ID+len(nodes_art)))
        for nid, xyz in zip(ids, nodes_art):
            node_xyz[nid] = xyz
        ID += len(nodes_art)
        D = distance_matrix(nodes_art, nodes_art)
        mst = minimum_spanning_tree(D).toarray().astype(float)
        s,t = np.where(mst)
        for si,ti in zip(s,t):
            a = ids[si]; b = ids[ti]
            all_arterial_segs.append((a,b))
            tree[a].append(b)
            diameters[a] = None
        diameters[ids[0]] = root_art_d

    # Venous MST (central vein)
    ven_nodes = np.vstack([central_vein.reshape(1,3), venous_points]) if len(venous_points)>0 else np.vstack([central_vein.reshape(1,3)])
    ven_ids = list(range(ID, ID+len(ven_nodes)))
    for nid, xyz in zip(ven_ids, ven_nodes):
        node_xyz[nid] = xyz
    ID += len(ven_nodes)
    Dv = distance_matrix(ven_nodes, ven_nodes)
    mst_v = minimum_spanning_tree(Dv).toarray().astype(float)
    s,t = np.where(mst_v)
    for si,ti in zip(s,t):
        a, b = ven_ids[si], ven_ids[ti]
        all_venous_segs.append((a,b))
        tree[a].append(b)
        diameters[a] = None
    diameters[ven_ids[0]] = root_ven_d

    # NOTE: capillaries intentionally omitted (user requested removal)
    capillary_segs = []

    # Murray's law (recursive assign) - compute diameters from leaves upward
    def assign_diameters(root, min_d=min_d):
        if not tree.get(root):
            diameters[root] = min_d
            return diameters[root]
        child_ds = []
        for child in tree[root]:
            d = assign_diameters(child, min_d)
            child_ds.append(d)
        parent_d = (sum([d**3 for d in child_ds]))**(1/3)
        diameters[root] = parent_d
        return parent_d

    # assign for all inlet roots
    for inlet_xyz in inlet_pts:
        root = next((k for k,v in node_xyz.items() if np.allclose(v, inlet_xyz)), None)
        if root is not None:
            assign_diameters(root)
    # assign for central vein root
    ven_root = next((k for k,v in node_xyz.items() if np.allclose(v, central_vein.reshape(3))), None)
    if ven_root is not None:
        assign_diameters(ven_root)

    # --- Ensure diameters exist and set user-specified root diameters ---
    for nid in node_xyz:
        if diameters.get(nid) is None:
            diameters[nid] = min_d

    # enforce inlet root diameters = root_art_d
    for ipt in inlet_pts:
        root_id = next((k for k,v in node_xyz.items() if np.allclose(v, ipt)), None)
        if root_id is not None:
            diameters[root_id] = root_art_d

    # enforce central vein root diameter
    if ven_root is not None:
        diameters[ven_root] = root_ven_d

    # Build KDTree of vessel nodes for nearest-diameter queries (used by plotting)
    v_ids = np.array(sorted(node_xyz.keys()))
    v_coords = np.array([node_xyz[k] for k in v_ids])
    v_diams  = np.array([diameters[k] for k in v_ids])
    if v_coords.shape[0] > 0:
        kdt_v = cKDTree(v_coords)
    else:
        kdt_v = None

else:
    print("Using existing network in memory (re-using variables).")
    # ensure min_d and root_art_d exist
    if 'min_d' not in globals(): min_d = 0.02
    if 'root_art_d' not in globals(): root_art_d = 8.0

# ---------- VEGF field (periportal baseline + hypoxia-induced term) ----------
# compute distance to nearest inlet for each attraction point
k_in = cKDTree(inlet_pts)
dist_to_inlet = k_in.query(attraction_points)[0]
max_dist = np.max(dist_to_inlet) if np.max(dist_to_inlet) > 0 else 1.0
base_vegf = 1.0 - (dist_to_inlet / max_dist)

# hypoxia term from O2
hypoxia_term = 1.0 - o_attract

# combine and normalize
alpha = 0.6; beta = 0.4
vegf_raw = alpha * base_vegf + beta * hypoxia_term
vegf_min, vegf_max = vegf_raw.min(), vegf_raw.max()
vegf = (vegf_raw - vegf_min) / max(1e-12, (vegf_max - vegf_min))

# ---------- Color maps ----------
blue_colors = [(2/255, 48/255, 71/255), (3/255, 70/255, 120/255), (30/255,100/255,180/255), (120/255,170/255,230/255), (180/255,210/255,250/255)]
cmap_o2 = LinearSegmentedColormap.from_list("dark2lightblue", blue_colors)
pink_colors = [(90/255, 15/255, 45/255), (140/255,35/255,90/255), (200/255,100/255,150/255), (240/255,160/255,200/255), (255/255,220/255,235/255)]
cmap_vegf = LinearSegmentedColormap.from_list("dark2lightpink", pink_colors)

# ---------- Save three static images ----------
os.makedirs("output_images", exist_ok=True)

# 1) oxygen only (attraction points only)
plt.figure(figsize=(8,8))
sc = plt.scatter(attraction_points[:,0], attraction_points[:,1], c=o_attract, cmap=cmap_o2, s=8)
plt.axis('equal'); plt.title("Attraction points colored by O₂")
plt.colorbar(sc, label="O₂ (normalized)")
plt.savefig("output_images/oxygen_only.png", dpi=200, bbox_inches='tight')
plt.close()

# 2) VEGF only
plt.figure(figsize=(8,8))
sc = plt.scatter(attraction_points[:,0], attraction_points[:,1], c=vegf, cmap=cmap_vegf, s=8)
plt.axis('equal'); plt.title("Attraction points colored by VEGF")
plt.colorbar(sc, label="VEGF (normalized)")
plt.savefig("output_images/vegf_only.png", dpi=200, bbox_inches='tight')
plt.close()

# 3) Combined: O2 | VEGF | Overlay
fig, axs = plt.subplots(1,3, figsize=(18,6))
ax = axs[0]
sc0 = ax.scatter(attraction_points[:,0], attraction_points[:,1], c=o_attract, cmap=cmap_o2, s=6)
ax.set_title("O₂"); ax.axis('equal'); plt.colorbar(sc0, ax=ax, fraction=0.046, pad=0.02)
ax = axs[1]
sc1 = ax.scatter(attraction_points[:,0], attraction_points[:,1], c=vegf, cmap=cmap_vegf, s=6)
ax.set_title("VEGF"); ax.axis('equal'); plt.colorbar(sc1, ax=ax, fraction=0.046, pad=0.02)
ax = axs[2]
ax.scatter(attraction_points[:,0], attraction_points[:,1], c=o_attract, cmap=cmap_o2, s=6, label="O2")
ax.scatter(attraction_points[:,0], attraction_points[:,1], c=vegf, cmap=cmap_vegf, s=12, alpha=0.33, label="VEGF")
ax.set_title("Overlay (O₂ + VEGF)"); ax.axis('equal'); axs[2].legend(loc='upper right')
plt.savefig("output_images/combined_oxygen_vegf.png", dpi=200, bbox_inches='tight')
plt.close()

print("Saved images: output_images/oxygen_only.png, vegf_only.png, combined_oxygen_vegf.png")

# ---------- 3D vessel Plotly visualization (no capillaries) ----------
your_mesh = mesh.Mesh.from_file(stl_filename)
verts, inv = np.unique(your_mesh.vectors.reshape(-1,3), axis=0, return_inverse=True)
faces = inv.reshape(-1,3)
i = faces[:,0]; j = faces[:,1]; k = faces[:,2]

fig = go.Figure()
fig.add_trace(go.Mesh3d(x=verts[:,0], y=verts[:,1], z=verts[:,2],
                       i=i,j=j,k=k, opacity=0.22, flatshading=True, name='Lobule surface'))

# helper to get diameter for node id or coord
def diam_at_coord(coord):
    if kdt_v is None: return float(min_d)
    _, idx = kdt_v.query(np.asarray(coord))
    return float(v_diams[idx])
def get_d_for_node_or_coord(x):
    if isinstance(x, (int, np.integer)):
        return float(diameters.get(int(x), min_d))
    try:
        if np.isscalar(x):
            return float(diameters.get(int(x), min_d))
    except Exception:
        pass
    return diam_at_coord(x)

# arterial traces
for (a,b) in all_arterial_segs:
    s = node_xyz[a]; e = node_xyz[b]
    da = get_d_for_node_or_coord(a)
    db = get_d_for_node_or_coord(b)
    lw = diam_to_lw_visual((da+db)/2.0)
    fig.add_trace(go.Scatter3d(x=[s[0], e[0]], y=[s[1], e[1]], z=[s[2], e[2]],
                               mode='lines', line=dict(color='crimson', width=lw), showlegend=False))

# venous traces
for (a,b) in all_venous_segs:
    s = node_xyz[a]; e = node_xyz[b]
    da = get_d_for_node_or_coord(a)
    db = get_d_for_node_or_coord(b)
    lw = diam_to_lw_visual((da+db)/2.0)
    fig.add_trace(go.Scatter3d(x=[s[0], e[0]], y=[s[1], e[1]], z=[s[2], e[2]],
                               mode='lines', line=dict(color='navy', width=lw), showlegend=False))

# attraction points colored by O2
fig.add_trace(go.Scatter3d(x=attraction_points[:,0], y=attraction_points[:,1], z=attraction_points[:,2],
                           mode='markers',
                           marker=dict(size=3, color=o_attract, colorscale='Blues', colorbar=dict(title='O₂')),
                           name='Attraction pts (O₂)'))

# overlay VEGF (transparent pink)
fig.add_trace(go.Scatter3d(x=attraction_points[:,0], y=attraction_points[:,1], z=attraction_points[:,2],
                           mode='markers',
                           marker=dict(size=6, color=vegf, colorscale=[[0,'#5a0f2d'],[0.5,'#c86496'],[1,'#ffdceb']], opacity=0.45),
                           name='VEGF (overlay)'))

# inlet and central markers
fig.add_trace(go.Scatter3d(x=inlet_pts[:,0], y=inlet_pts[:,1], z=inlet_pts[:,2],
                           mode='markers', marker=dict(size=6, symbol='diamond', color='red'), name='Inlets'))
fig.add_trace(go.Scatter3d(x=[central_vein[0,0]], y=[central_vein[0,1]], z=[central_vein[0,2]],
                           mode='markers', marker=dict(size=8, symbol='circle', color='blue'), name='Central vein'))

fig.update_layout(scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
                  width=1000, height=820)
fig.show()




ModuleNotFoundError: No module named 'stl'

In [ ]:
!pip install numpy-stl


In [ ]:
from stl import mesh



In [ ]:
!pip install numpy numpy-stl scipy matplotlib plotly imageio imageio-ffmpeg kaleido


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.5 MB/s eta 0:00:00


In [ ]:
# === VEGF + Murray fixes + 3 PNGs + 3D vessel plot (capillaries removed) ===
# Paste this cell in Colab / Jupyter and run. It will rebuild network if needed.
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import plotly.graph_objects as go
from stl import mesh
from scipy.spatial import cKDTree, ConvexHull
from scipy.spatial import distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from matplotlib.path import Path
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve
import collections

# ---------- PARAMETERS ----------
stl_filename = "liver_lobule.stl"
n_attraction = 900
knn = 8
min_d = 0.02
root_art_d = 8.0    # user-requested root inlet diameter (mm)
root_ven_d = 8.0
# visualization scaling
def diam_to_lw_visual(d, dmin=min_d, dmax=root_art_d):
    t = (d - dmin) / max(1e-9, (dmax - dmin))
    t = np.clip(t, 0.0, 1.0)
    return 0.8 + 18.0 * (t ** 0.6)

# ---------- Build network if not present ----------
need_build = ('node_xyz' not in globals()) or ('all_arterial_segs' not in globals())
if need_build:
    print("Building network (this can take a short while)...")
    your_mesh = mesh.Mesh.from_file(stl_filename)
    all_points = your_mesh.vectors.reshape(-1, 3)
    z_top = np.max(all_points[:, 2]); z_bottom = np.min(all_points[:, 2])
    top_points = all_points[all_points[:, 2] > np.percentile(all_points[:, 2], 95) - 1e-2]
    xy = top_points[:, :2]
    centroid = xy.mean(axis=0)
    hull = ConvexHull(xy)
    hex_boundary = xy[hull.vertices]
    polygon = Path(hex_boundary)

    # Exactly 6 inlets (angular bins)
    vecs = xy - centroid
    angles = np.arctan2(vecs[:,1], vecs[:,0])
    r = np.linalg.norm(vecs, axis=1)
    bin_edges = np.linspace(-np.pi, np.pi, 7)
    bin_idx = np.digitize(angles, bin_edges) - 1
    bin_idx = np.mod(bin_idx, 6)

    inlet_pts = []
    for b in range(6):
        inds = np.where(bin_idx == b)[0]
        if inds.size == 0:
            center_angle = 0.5*(bin_edges[b] + bin_edges[b+1])
            ang_diff = np.abs(np.angle(np.exp(1j*(angles - center_angle))))
            chosen = np.argmin(ang_diff)
        else:
            chosen_local = inds[np.argmax(r[inds])]
            chosen = chosen_local
        inlet_pts.append(top_points[chosen])
    inlet_pts = np.array(inlet_pts)
    central_vein = np.array([centroid[0], centroid[1], z_bottom]).reshape(1,3)

    # attraction points inside hex
    def scatter_points(n):
        min_x, min_y = np.min(hex_boundary, axis=0)
        max_x, max_y = np.max(hex_boundary, axis=0)
        pts=[]
        while len(pts) < n:
            x = np.random.uniform(min_x, max_x); y = np.random.uniform(min_y, max_y)
            if not polygon.contains_point([x,y]): continue
            z = np.random.uniform(z_bottom, z_top)
            pts.append([x,y,z])
        return np.array(pts)

    attraction_points = scatter_points(n_attraction)

    # Build Laplacian & O2
    nodes = np.vstack([inlet_pts, central_vein, attraction_points])
    n_inlets = len(inlet_pts)
    idx_inlets = np.arange(n_inlets)
    idx_central = n_inlets
    idx_attract = np.arange(n_inlets+1, nodes.shape[0])

    kdt = cKDTree(nodes)
    dists, neigh = kdt.query(nodes, k=knn+1)
    rows=[]; cols=[]; data=[]
    for i in range(nodes.shape[0]):
        for j in neigh[i,1:]:
            dij = np.linalg.norm(nodes[i]-nodes[j])
            if dij == 0: w = 0.0
            else: w = 1.0/(dij + 1e-12)
            rows.append(i); cols.append(j); data.append(w)
    rows_sym = rows + cols; cols_sym = cols + rows; data_sym = data + data
    A = csr_matrix((data_sym, (rows_sym, cols_sym)), shape=(nodes.shape[0], nodes.shape[0]))
    deg = np.array(A.sum(axis=1)).flatten()
    D = csr_matrix((deg, (np.arange(nodes.shape[0]), np.arange(nodes.shape[0]))), shape=(nodes.shape[0], nodes.shape[0]))
    L = D - A
    boundary_idx = np.concatenate([idx_inlets, [idx_central]])
    interior_idx = np.setdiff1d(np.arange(nodes.shape[0]), boundary_idx)
    L_uu = L[np.ix_(interior_idx, interior_idx)]
    L_ub = L[np.ix_(interior_idx, boundary_idx)]
    o_b = np.concatenate([np.ones(len(idx_inlets)), np.zeros(1)])
    o_u = spsolve(L_uu, - L_ub.dot(o_b))
    O = np.zeros(nodes.shape[0])
    O[boundary_idx] = o_b
    O[interior_idx] = o_u
    o_attract = O[idx_attract]

    # Exclusive assign to nearest root (inlets or central)
    k_roots = cKDTree(np.vstack([inlet_pts, central_vein]))
    droot, iroot = k_roots.query(attraction_points)
    assigned_to_inlet = (iroot < n_inlets)
    arterial_points = attraction_points[assigned_to_inlet]
    venous_points   = attraction_points[~assigned_to_inlet]
    arterial_root_idx_for_point = iroot[assigned_to_inlet]

    # Build arterial MSTs (one per inlet)
    node_xyz = {}
    ID = 0
    all_arterial_segs = []
    all_venous_segs = []
    tree = collections.defaultdict(list)
    diameters = {}

    for inlet_idx in range(n_inlets):
        mask = (arterial_root_idx_for_point == inlet_idx)
        group_pts = arterial_points[mask] if len(arterial_points)>0 else np.empty((0,3))
        nodes_art = np.vstack([inlet_pts[inlet_idx].reshape(1,3), group_pts]) if len(group_pts)>0 else np.vstack([inlet_pts[inlet_idx].reshape(1,3)])
        ids = list(range(ID, ID+len(nodes_art)))
        for nid, xyz in zip(ids, nodes_art):
            node_xyz[nid] = xyz
        ID += len(nodes_art)
        D = distance_matrix(nodes_art, nodes_art)
        mst = minimum_spanning_tree(D).toarray().astype(float)
        s,t = np.where(mst)
        for si,ti in zip(s,t):
            a = ids[si]; b = ids[ti]
            all_arterial_segs.append((a,b))
            tree[a].append(b)
            diameters[a] = None
        diameters[ids[0]] = root_art_d

    # Venous MST (central vein)
    ven_nodes = np.vstack([central_vein.reshape(1,3), venous_points]) if len(venous_points)>0 else np.vstack([central_vein.reshape(1,3)])
    ven_ids = list(range(ID, ID+len(ven_nodes)))
    for nid, xyz in zip(ven_ids, ven_nodes):
        node_xyz[nid] = xyz
    ID += len(ven_nodes)
    Dv = distance_matrix(ven_nodes, ven_nodes)
    mst_v = minimum_spanning_tree(Dv).toarray().astype(float)
    s,t = np.where(mst_v)
    for si,ti in zip(s,t):
        a, b = ven_ids[si], ven_ids[ti]
        all_venous_segs.append((a,b))
        tree[a].append(b)
        diameters[a] = None
    diameters[ven_ids[0]] = root_ven_d

    # NOTE: capillaries intentionally omitted (user requested removal)
    capillary_segs = []

    # Murray's law (recursive assign) - compute diameters from leaves upward
    def assign_diameters(root, min_d=min_d):
        if not tree.get(root):
            diameters[root] = min_d
            return diameters[root]
        child_ds = []
        for child in tree[root]:
            d = assign_diameters(child, min_d)
            child_ds.append(d)
        parent_d = (sum([d**3 for d in child_ds]))**(1/3)
        diameters[root] = parent_d
        return parent_d

    # assign for all inlet roots
    for inlet_xyz in inlet_pts:
        root = next((k for k,v in node_xyz.items() if np.allclose(v, inlet_xyz)), None)
        if root is not None:
            assign_diameters(root)
    # assign for central vein root
    ven_root = next((k for k,v in node_xyz.items() if np.allclose(v, central_vein.reshape(3))), None)
    if ven_root is not None:
        assign_diameters(ven_root)

    # --- Ensure diameters exist and set user-specified root diameters ---
    for nid in node_xyz:
        if diameters.get(nid) is None:
            diameters[nid] = min_d

    # enforce inlet root diameters = root_art_d
    for ipt in inlet_pts:
        root_id = next((k for k,v in node_xyz.items() if np.allclose(v, ipt)), None)
        if root_id is not None:
            diameters[root_id] = root_art_d

    # enforce central vein root diameter
    if ven_root is not None:
        diameters[ven_root] = root_ven_d

    # Build KDTree of vessel nodes for nearest-diameter queries (used by plotting)
    v_ids = np.array(sorted(node_xyz.keys()))
    v_coords = np.array([node_xyz[k] for k in v_ids])
    v_diams  = np.array([diameters[k] for k in v_ids])
    if v_coords.shape[0] > 0:
        kdt_v = cKDTree(v_coords)
    else:
        kdt_v = None

else:
    print("Using existing network in memory (re-using variables).")
    # ensure min_d and root_art_d exist
    if 'min_d' not in globals(): min_d = 0.02
    if 'root_art_d' not in globals(): root_art_d = 8.0

# ---------- VEGF field (periportal baseline + hypoxia-induced term) ----------
# compute distance to nearest inlet for each attraction point
k_in = cKDTree(inlet_pts)
dist_to_inlet = k_in.query(attraction_points)[0]
max_dist = np.max(dist_to_inlet) if np.max(dist_to_inlet) > 0 else 1.0
base_vegf = 1.0 - (dist_to_inlet / max_dist)

# hypoxia term from O2
hypoxia_term = 1.0 - o_attract

# combine and normalize
alpha = 0.6; beta = 0.4
vegf_raw = alpha * base_vegf + beta * hypoxia_term
vegf_min, vegf_max = vegf_raw.min(), vegf_raw.max()
vegf = (vegf_raw - vegf_min) / max(1e-12, (vegf_max - vegf_min))

# ---------- Color maps ----------
blue_colors = [(2/255, 48/255, 71/255), (3/255, 70/255, 120/255), (30/255,100/255,180/255), (120/255,170/255,230/255), (180/255,210/255,250/255)]
cmap_o2 = LinearSegmentedColormap.from_list("dark2lightblue", blue_colors)
pink_colors = [(90/255, 15/255, 45/255), (140/255,35/255,90/255), (200/255,100/255,150/255), (240/255,160/255,200/255), (255/255,220/255,235/255)]
cmap_vegf = LinearSegmentedColormap.from_list("dark2lightpink", pink_colors)

# ---------- Save three static images ----------
os.makedirs("output_images", exist_ok=True)

# 1) oxygen only (attraction points only)
plt.figure(figsize=(8,8))
sc = plt.scatter(attraction_points[:,0], attraction_points[:,1], c=o_attract, cmap=cmap_o2, s=8)
plt.axis('equal'); plt.title("Attraction points colored by O₂")
plt.colorbar(sc, label="O₂ (normalized)")
plt.savefig("output_images/oxygen_only.png", dpi=200, bbox_inches='tight')
plt.close()

# 2) VEGF only
plt.figure(figsize=(8,8))
sc = plt.scatter(attraction_points[:,0], attraction_points[:,1], c=vegf, cmap=cmap_vegf, s=8)
plt.axis('equal'); plt.title("Attraction points colored by VEGF")
plt.colorbar(sc, label="VEGF (normalized)")
plt.savefig("output_images/vegf_only.png", dpi=200, bbox_inches='tight')
plt.close()

# 3) Combined: O2 | VEGF | Overlay
fig, axs = plt.subplots(1,3, figsize=(18,6))
ax = axs[0]
sc0 = ax.scatter(attraction_points[:,0], attraction_points[:,1], c=o_attract, cmap=cmap_o2, s=6)
ax.set_title("O₂"); ax.axis('equal'); plt.colorbar(sc0, ax=ax, fraction=0.046, pad=0.02)
ax = axs[1]
sc1 = ax.scatter(attraction_points[:,0], attraction_points[:,1], c=vegf, cmap=cmap_vegf, s=6)
ax.set_title("VEGF"); ax.axis('equal'); plt.colorbar(sc1, ax=ax, fraction=0.046, pad=0.02)
ax = axs[2]
ax.scatter(attraction_points[:,0], attraction_points[:,1], c=o_attract, cmap=cmap_o2, s=6, label="O2")
ax.scatter(attraction_points[:,0], attraction_points[:,1], c=vegf, cmap=cmap_vegf, s=12, alpha=0.33, label="VEGF")
ax.set_title("Overlay (O₂ + VEGF)"); ax.axis('equal'); axs[2].legend(loc='upper right')
plt.savefig("output_images/combined_oxygen_vegf.png", dpi=200, bbox_inches='tight')
plt.close()

print("Saved images: output_images/oxygen_only.png, vegf_only.png, combined_oxygen_vegf.png")

# ---------- 3D vessel Plotly visualization (no capillaries) ----------
your_mesh = mesh.Mesh.from_file(stl_filename)
verts, inv = np.unique(your_mesh.vectors.reshape(-1,3), axis=0, return_inverse=True)
faces = inv.reshape(-1,3)
i = faces[:,0]; j = faces[:,1]; k = faces[:,2]

fig = go.Figure()
fig.add_trace(go.Mesh3d(x=verts[:,0], y=verts[:,1], z=verts[:,2],
                       i=i,j=j,k=k, opacity=0.22, flatshading=True, name='Lobule surface'))

# helper to get diameter for node id or coord
def diam_at_coord(coord):
    if kdt_v is None: return float(min_d)
    _, idx = kdt_v.query(np.asarray(coord))
    return float(v_diams[idx])
def get_d_for_node_or_coord(x):
    if isinstance(x, (int, np.integer)):
        return float(diameters.get(int(x), min_d))
    try:
        if np.isscalar(x):
            return float(diameters.get(int(x), min_d))
    except Exception:
        pass
    return diam_at_coord(x)

# arterial traces
for (a,b) in all_arterial_segs:
    s = node_xyz[a]; e = node_xyz[b]
    da = get_d_for_node_or_coord(a)
    db = get_d_for_node_or_coord(b)
    lw = diam_to_lw_visual((da+db)/2.0)
    fig.add_trace(go.Scatter3d(x=[s[0], e[0]], y=[s[1], e[1]], z=[s[2], e[2]],
                               mode='lines', line=dict(color='crimson', width=lw), showlegend=False))

# venous traces
for (a,b) in all_venous_segs:
    s = node_xyz[a]; e = node_xyz[b]
    da = get_d_for_node_or_coord(a)
    db = get_d_for_node_or_coord(b)
    lw = diam_to_lw_visual((da+db)/2.0)
    fig.add_trace(go.Scatter3d(x=[s[0], e[0]], y=[s[1], e[1]], z=[s[2], e[2]],
                               mode='lines', line=dict(color='navy', width=lw), showlegend=False))

# attraction points colored by O2
fig.add_trace(go.Scatter3d(x=attraction_points[:,0], y=attraction_points[:,1], z=attraction_points[:,2],
                           mode='markers',
                           marker=dict(size=3, color=o_attract, colorscale='Blues', colorbar=dict(title='O₂')),
                           name='Attraction pts (O₂)'))

# overlay VEGF (transparent pink)
fig.add_trace(go.Scatter3d(x=attraction_points[:,0], y=attraction_points[:,1], z=attraction_points[:,2],
                           mode='markers',
                           marker=dict(size=6, color=vegf, colorscale=[[0,'#5a0f2d'],[0.5,'#c86496'],[1,'#ffdceb']], opacity=0.45),
                           name='VEGF (overlay)'))

# inlet and central markers
fig.add_trace(go.Scatter3d(x=inlet_pts[:,0], y=inlet_pts[:,1], z=inlet_pts[:,2],
                           mode='markers', marker=dict(size=6, symbol='diamond', color='red'), name='Inlets'))
fig.add_trace(go.Scatter3d(x=[central_vein[0,0]], y=[central_vein[0,1]], z=[central_vein[0,2]],
                           mode='markers', marker=dict(size=8, symbol='circle', color='blue'), name='Central vein'))

fig.update_layout(scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
                  width=1000, height=820)
fig.show()




Building network (this can take a short while)...


FileNotFoundError: [Errno 2] No such file or directory: 'liver_lobule.stl'

# NEW

In [ ]:
import sys
if 'google.colab' in sys.modules:
  %pip install --quiet numpy-stl

"""
Liver lobule — VEGF-guided angiogenesis simulation
====================================================
Extends the original O2/MST model with:
  1.  Dual diffusion fields: O2 (Laplacian) + VEGF (= 1 - O2, clamped by threshold)
  2.  Angiogenic tip-cell sprouting from leaf nodes of existing vasculature
  3.  Chemotactic random walk: tip cells climb ∇VEGF toward hypoxic zones
  4.  Stalk segment recording: each tip-cell step becomes a new capillary edge
  5.  Anastomosis: tip cells that come within snap_r of any vessel node connect to it
  6.  Iterative feedback: O2 field is re-solved after new segments are added (n_iter loops)
  7.  Plotly visualisation: new angiogenic segments shown in green; original capillaries gold;
      attraction pts coloured by VEGF (plasma colorscale)

Run in Colab/Jupyter after uploading liver_lobule.stl
"""

import numpy as np
from stl import mesh
from scipy.spatial import cKDTree, ConvexHull, distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve
from matplotlib.path import Path
import plotly.graph_objects as go
import collections, warnings
warnings.filterwarnings('ignore')

# ─────────────────────────── PARAMETERS ────────────────────────────────────
# stl_filename   = "liver_lobule.stl"  # Removed hardcoded filename
n_attraction   = 700
knn            = 8
min_d          = 0.02          # terminal diameter (mm)
root_art_d     = 0.14
root_ven_d     = 0.13

# *** NEW *** angiogenesis parameters
o2_threshold   = 0.35          # O2 < this → hypoxic → VEGF source
n_sprouts      = 12            # tip cells to launch per iteration
walk_steps     = 30            # steps each tip cell takes
step_size      = 0.04          # mm per step
bias_strength  = 0.65          # 0=random walk  1=pure gradient climb
branch_prob    = 0.12          # probability of branching at each step
snap_r         = 0.07          # anastomosis snap radius (mm)
n_iter         = 3             # feedback iterations (re-solve O2 each time)

np.random.seed(42)

# ─────────────────────────── LOAD MESH & HEX ────────────────────────────────
your_mesh  = mesh.Mesh.from_file(stl_filename)
all_points = your_mesh.vectors.reshape(-1, 3)
z_top      = np.max(all_points[:, 2])
z_bottom   = np.min(all_points[:, 2])

top_points = all_points[all_points[:, 2] > np.percentile(all_points[:, 2], 95) - 1e-2]
xy         = top_points[:, :2]
centroid   = xy.mean(axis=0)
hull       = ConvexHull(xy)
hex_boundary = xy[hull.vertices]
polygon    = Path(hex_boundary)

# ── 6 inlet positions (one per angular bin) ─────────────────────────────────
vecs   = xy - centroid
angles = np.arctan2(vecs[:, 1], vecs[:, 0])
r      = np.linalg.norm(vecs, axis=1)
bin_edges = np.linspace(-np.pi, np.pi, 7)
bin_idx   = np.mod(np.digitize(angles, bin_edges) - 1, 6)

inlet_pts = []
for b in range(6):
    inds = np.where(bin_idx == b)[0]
    if inds.size == 0:
        center_angle = 0.5 * (bin_edges[b] + bin_edges[b+1])
        ang_diff = np.abs(np.angle(np.exp(1j * (angles - center_angle))))
        chosen = np.argmin(ang_diff)
    else:
        chosen = inds[np.argmax(r[inds])]
    inlet_pts.append(top_points[chosen])
inlet_pts = np.array(inlet_pts)

used = set()
for i in range(len(inlet_pts)):
    tup = tuple(np.round(inlet_pts[i], 8))
    if tup in used:
        b = i; inds = np.where(bin_idx == b)[0]
        sorted_inds = inds[np.argsort(-r[inds])]
        for si in sorted_inds:
            cand = tuple(np.round(top_points[si], 8))
            if cand not in used:
                inlet_pts[i] = top_points[si]; used.add(cand); break
    else:
        used.add(tup)

central_vein = np.array([centroid[0], centroid[1], z_bottom]).reshape(1, 3)

# ── Attraction points ────────────────────────────────────────────────────────
def scatter_points(n):
    min_x, min_y = np.min(hex_boundary, axis=0)
    max_x, max_y = np.max(hex_boundary, axis=0)
    pts = []
    while len(pts) < n:
        x = np.random.uniform(min_x, max_x)
        y = np.random.uniform(min_y, max_y)
        if not polygon.contains_point([x, y]): continue
        z = np.random.uniform(z_bottom, z_top)
        pts.append([x, y, z])
    return np.array(pts)

attraction_points = scatter_points(n_attraction)
n_inlets  = len(inlet_pts)
idx_inlets  = np.arange(0, n_inlets)
idx_central = n_inlets

# ── Build k-NN Laplacian ─────────────────────────────────────────────────────
def build_laplacian(nodes):
    kdt = cKDTree(nodes)
    dists, neigh = kdt.query(nodes, k=knn + 1)
    rows, cols, data = [], [], []
    for i in range(nodes.shape[0]):
        for j in neigh[i, 1:]:
            dij = np.linalg.norm(nodes[i] - nodes[j])
            w = 1.0 / (dij + 1e-12)
            rows.append(i); cols.append(j); data.append(w)
    rows_sym = rows + cols; cols_sym = cols + rows; data_sym = data + data
    A   = csr_matrix((data_sym, (rows_sym, cols_sym)), shape=(nodes.shape[0],) * 2)
    deg = np.array(A.sum(axis=1)).flatten()
    D   = csr_matrix((deg, (np.arange(nodes.shape[0]),) * 2), shape=A.shape)
    return D - A

nodes = np.vstack([inlet_pts, central_vein, attraction_points])
idx_attract = np.arange(n_inlets + 1, nodes.shape[0])
boundary_idx = np.concatenate([idx_inlets, [idx_central]])
interior_idx = np.setdiff1d(np.arange(nodes.shape[0]), boundary_idx)
o_b          = np.concatenate([np.ones(n_inlets), np.zeros(1)])

def solve_o2(nodes, boundary_idx, interior_idx, o_b):
    L    = build_laplacian(nodes)
    L_uu = L[np.ix_(interior_idx, interior_idx)]
    L_ub = L[np.ix_(interior_idx, boundary_idx)]
    o_u  = spsolve(L_uu, -L_ub.dot(o_b))
    O    = np.zeros(nodes.shape[0])
    O[boundary_idx] = o_b
    O[interior_idx] = o_u
    return O

O         = solve_o2(nodes, boundary_idx, interior_idx, o_b)
o_attract = O[idx_attract]

# ── Assignment & MSTs (unchanged) ───────────────────────────────────────────
k_roots = cKDTree(np.vstack([inlet_pts, central_vein]))
droot, iroot = k_roots.query(attraction_points)
assigned_to_inlet = (iroot < n_inlets)
arterial_points   = attraction_points[assigned_to_inlet]
venous_points     = attraction_points[~assigned_to_inlet]
arterial_root_idx = iroot[assigned_to_inlet]

node_xyz = {}
ID = 0
all_arterial_segs = []
all_venous_segs   = []
tree = collections.defaultdict(list)
diameters = {}

for inlet_idx in range(n_inlets):
    mask      = (arterial_root_idx == inlet_idx)
    group_pts = arterial_points[mask] if mask.any() else np.empty((0, 3))
    nodes_art = np.vstack([inlet_pts[inlet_idx].reshape(1, 3), group_pts]) if len(group_pts) else inlet_pts[inlet_idx].reshape(1, 3)
    ids = list(range(ID, ID + len(nodes_art)))
    for nid, xyz in zip(ids, nodes_art):
        node_xyz[nid] = xyz
    ID += len(nodes_art)
    D_mat = distance_matrix(nodes_art, nodes_art)
    mst   = minimum_spanning_tree(D_mat).toarray()
    for si, ti in zip(*np.where(mst)):
        a, b = ids[si], ids[ti]
        all_arterial_segs.append((a, b))
        tree[a].append(b)
        diameters[a] = None
    diameters[ids[0]] = root_art_d

ven_nodes = np.vstack([central_vein.reshape(1, 3), venous_points]) if len(venous_points) else central_vein.reshape(1, 3)
ven_ids   = list(range(ID, ID + len(ven_nodes)))
for nid, xyz in zip(ven_ids, ven_nodes):
    node_xyz[nid] = xyz
ID += len(ven_nodes)
Dv    = distance_matrix(ven_nodes, ven_nodes)
mst_v = minimum_spanning_tree(Dv).toarray()
for si, ti in zip(*np.where(mst_v)):
    a, b = ven_ids[si], ven_ids[ti]
    all_venous_segs.append((a, b))
    tree[a].append(b)
    diameters[a] = None
diameters[ven_ids[0]] = root_ven_d

# original capillaries
def leaves(node_set):
    return [n for n in node_set if len(tree.get(n, [])) == 0]

arterial_node_set = set(a for seg in all_arterial_segs for a in seg)
venous_node_set   = set(a for seg in all_venous_segs   for a in seg)
venous_node_set.update(ven_ids[:1])
arterial_leaves   = leaves(arterial_node_set)
venous_leaves     = leaves(venous_node_set) or list(venous_node_set)
vleaf_coords      = np.array([node_xyz[n] for n in venous_leaves])
k_vleaf           = cKDTree(vleaf_coords) if len(vleaf_coords) else None

capillary_segs = []
for a in arterial_leaves:
    if k_vleaf is None: break
    d, idx = k_vleaf.query(node_xyz[a])
    v_global = venous_leaves[idx]
    if np.linalg.norm(node_xyz[a] - node_xyz[v_global]) < 1e-9: continue
    capillary_segs.append((a, v_global))
    tree[a].append(v_global)

# Murray's law
def assign_diameters(root, min_d=min_d):
    if not tree.get(root):
        diameters[root] = min_d; return min_d
    child_ds = [assign_diameters(c, min_d) for c in tree[root]]
    d = (sum(c**3 for c in child_ds))**(1/3)
    diameters[root] = d; return d

for xyz in inlet_pts:
    root = next((k for k, v in node_xyz.items() if np.allclose(v, xyz)), None)
    if root is not None: assign_diameters(root)
ven_root = next((k for k, v in node_xyz.items() if np.allclose(v, central_vein.reshape(3))), None)
if ven_root is not None: assign_diameters(ven_root)

# ─────────────────── *** NEW *** VEGF-GUIDED ANGIOGENESIS ────────────────────

all_vessel_coords = np.array(list(node_xyz.values()))   # live snapshot (grows)
all_vessel_ids    = list(node_xyz.keys())                # parallel list

angio_segs        = []   # new angiogenic segments

def vegf_field(o2_values, threshold=o2_threshold):
    """VEGF = max(0, threshold - O2) normalised to [0,1]"""
    v = np.clip(threshold - o2_values, 0, None)
    vmax = v.max()
    return v / vmax if vmax > 1e-9 else v

def vegf_gradient_at(pos, attract_xyz, vegf_vals, knn_v=12):
    """Numerical ∇VEGF at pos via weighted average of nearby attraction points."""
    k     = min(knn_v, len(attract_xyz))
    dists = np.linalg.norm(attract_xyz - pos, axis=1)
    idx   = np.argpartition(dists, k)[:k]
    w     = 1.0 / (dists[idx] + 1e-9)
    diff  = attract_xyz[idx] - pos   # shape (k,3)
    grad  = (w[:, None] * diff * vegf_vals[idx, None]).sum(axis=0)
    norm  = np.linalg.norm(grad)
    return grad / norm if norm > 1e-9 else np.random.randn(3)

def clamp_to_lobule(pos):
    """Keep point inside lobule bounding box (rough clamping)."""
    min_x, min_y = np.min(hex_boundary, axis=0)
    max_x, max_y = np.max(hex_boundary, axis=0)
    pos[0] = np.clip(pos[0], min_x, max_x)
    pos[1] = np.clip(pos[1], min_y, max_y)
    pos[2] = np.clip(pos[2], z_bottom, z_top)
    return pos

for iteration in range(n_iter):
    # ── 1. Re-solve O2 (boundary now includes angiogenic nodes) ──────────────
    # Add new vessel nodes as additional O2 sources (value = 0.6, reduced supply)
    if iteration > 0 and len(angio_segs):
        angio_node_ids_in_node_xyz = list({nid for seg in angio_segs for nid in seg})
        angio_coords = np.array([node_xyz[nid] for nid in angio_node_ids_in_node_xyz])
        nodes_ext    = np.vstack([nodes, angio_coords])
        new_b_idx    = np.arange(len(nodes), len(nodes_ext))
        boundary_ext = np.concatenate([boundary_idx, new_b_idx])
        interior_ext = np.setdiff1d(np.arange(len(nodes_ext)), boundary_ext)
        o_b_ext      = np.concatenate([o_b, np.full(len(new_b_idx), 0.6)])
        O_ext        = solve_o2(nodes_ext, boundary_ext, interior_ext, o_b_ext)
        o_attract    = O_ext[idx_attract]

    vegf_attract = vegf_field(o_attract)
    attract_xyz  = attraction_points

    # ── 2. Choose sprout initiation sites (vessel leaves near hypoxic zones) ──
    hypoxic_mask = (o_attract < o2_threshold)
    if hypoxic_mask.sum() < 3:
        print(f"Iter {iteration}: no hypoxic zones, skipping sprouting."); continue

    hypoxic_pts  = attract_xyz[hypoxic_mask]
    k_hyp        = cKDTree(hypoxic_pts)

    # find vessel nodes closest to hypoxic zones → sprout origins
    all_v_coords_now = np.array([node_xyz[nid] for nid in node_xyz])
    dh, _ = k_hyp.query(all_v_coords_now)
    sorted_vessel_ids = [list(node_xyz.keys())[i] for i in np.argsort(dh)]
    # pick top-n_sprouts unique starts (avoid overlapping sprouts)
    starts_chosen = []; used_start_set = set()
    for vid in sorted_vessel_ids:
        tkey = tuple(np.round(node_xyz[vid], 4))
        if tkey not in used_start_set:
            starts_chosen.append(vid)
            used_start_set.add(tkey)
        if len(starts_chosen) >= n_sprouts: break

    # ── 3. Tip-cell chemotactic walk ──────────────────────────────────────────
    tip_queue = [(vid, 0) for vid in starts_chosen]  # (parent_node_id, depth)
    while tip_queue:
        parent_id, depth = tip_queue.pop(0)
        if depth >= walk_steps: continue

        pos       = node_xyz[parent_id].copy().astype(float)
        grad      = vegf_gradient_at(pos, attract_xyz, vegf_attract)
        rand_dir  = np.random.randn(3); rand_dir /= (np.linalg.norm(rand_dir) + 1e-9)
        direction = bias_strength * grad + (1 - bias_strength) * rand_dir
        d_norm    = np.linalg.norm(direction)
        if d_norm < 1e-9: direction = rand_dir; d_norm = 1.0
        direction /= d_norm

        new_pos   = clamp_to_lobule(pos + step_size * direction)

        # ── 4. Anastomosis check ─────────────────────────────────────────────
        all_v_arr  = np.array([node_xyz[nid] for nid in node_xyz])
        near_dists = np.linalg.norm(all_v_arr - new_pos, axis=1)
        closest_id = list(node_xyz.keys())[np.argmin(near_dists)]
        if near_dists.min() < snap_r and closest_id != parent_id:
            # anastomose: connect tip to existing vessel
            if (parent_id, closest_id) not in set(angio_segs) and \
               (closest_id, parent_id) not in set(angio_segs):
                angio_segs.append((parent_id, closest_id))
            continue  # tip cell consumed

        # ── 5. Record new stalk segment ──────────────────────────────────────
        new_id          = ID; ID += 1
        node_xyz[new_id] = new_pos
        diameters[new_id] = min_d
        angio_segs.append((parent_id, new_id))
        tree[parent_id].append(new_id)

        # ── 6. Branching ──────────────────────────────────────────────────────
        if np.random.rand() < branch_prob and depth < walk_steps - 5:
            tip_queue.append((new_id, depth + 1))  # branch tip
        tip_queue.append((new_id, depth + 1))       # main tip continues

    print(f"Iter {iteration+1}/{n_iter}: "
          f"{hypoxic_mask.sum()} hypoxic pts | "
          f"{len(starts_chosen)} sprouts | "
          f"{len(angio_segs)} angio segs total")

print(f"\nTotal angiogenic segments: {len(angio_segs)}")

# ─────────────────────────── VISUALISATION ──────────────────────────────────
def diam_to_lw(d, dmin=min_d, dmax=root_art_d):
    t = (d - dmin) / max(1e-9, dmax - dmin)
    return 0.6 + 7.4 * np.clip(t, 0, 1)

def seg_traces(segs, color, name, width_scale=1.0, dash='solid'):
    traces = []
    for idx_s, (a, b) in enumerate(segs):
        s  = node_xyz[a]; e = node_xyz[b]
        da = diameters.get(a, min_d) or min_d
        db = diameters.get(b, min_d) or min_d
        lw = diam_to_lw((da + db) / 2.0) * width_scale
        traces.append(go.Scatter3d(
            x=[s[0], e[0]], y=[s[1], e[1]], z=[s[2], e[2]],
            mode='lines',
            line=dict(color=color, width=lw, dash=dash),
            showlegend=(idx_s == 0), name=name if idx_s == 0 else None))
    return traces

verts, inv = np.unique(your_mesh.vectors.reshape(-1, 3), axis=0, return_inverse=True)
faces = inv.reshape(-1, 3)
i_f, j_f, k_f = faces[:, 0], faces[:, 1], faces[:, 2]

fig = go.Figure()
fig.add_trace(go.Mesh3d(x=verts[:,0], y=verts[:,1], z=verts[:,2],
                         i=i_f, j=j_f, k=k_f,
                         opacity=0.18, flatshading=True, name='Lobule surface'))

for tr in seg_traces(all_arterial_segs, 'crimson',  'Arterial (inlets)'):     fig.add_trace(tr)
for tr in seg_traces(all_venous_segs,   'navy',     'Venous (central)'):      fig.add_trace(tr)
for tr in seg_traces(capillary_segs,    'gold',     'Capillaries', 1.6):      fig.add_trace(tr)
for tr in seg_traces(angio_segs,        'limegreen','Angiogenic sprouts', 1.4):fig.add_trace(tr)  # *** NEW ***

fig.add_trace(go.Scatter3d(x=inlet_pts[:,0], y=inlet_pts[:,1], z=inlet_pts[:,2],
                            mode='markers',
                            marker=dict(size=6, symbol='diamond', color='red'),
                            name='Inlets (6)'))
fig.add_trace(go.Scatter3d(x=[central_vein[0,0]], y=[central_vein[0,1]], z=[central_vein[0,2]],
                            mode='markers',
                            marker=dict(size=7, symbol='circle', color='blue'),
                            name='Central vein'))

# *** NEW *** Colour attraction pts by VEGF (plasma scale) instead of O2
vegf_attract_final = vegf_field(o_attract)
fig.add_trace(go.Scatter3d(
    x=attraction_points[:,0], y=attraction_points[:,1], z=attraction_points[:,2],
    mode='markers',
    marker=dict(size=3, color=vegf_attract_final,
                colorscale='Plasma',
                colorbar=dict(title='VEGF'), showscale=True),
    name='Attraction pts (VEGF)'))

fig.update_layout(
    scene=dict(aspectmode='data',
               xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)'),
    width=1050, height=850,
    title='Liver lobule — VEGF-guided angiogenesis',
    legend=dict(x=0.01, y=0.99))
fig.show()

# ── Diagnostics ──────────────────────────────────────────────────────────────
print("\nInlet coords (6):")
for i, c in enumerate(inlet_pts): print(f"  {i}: {c}")
print(f"Central vein: {central_vein.reshape(3)}")
print(f"Arterial segs : {len(all_arterial_segs)}")
print(f"Venous segs   : {len(all_venous_segs)}")
print(f"Capillary segs: {len(capillary_segs)}")
print(f"Angio segs    : {len(angio_segs)}")          # *** NEW ***
print(f"Hypoxic frac  : {(o_attract < o2_threshold).mean():.2%}")  # *** NEW ***

Iter 0: no hypoxic zones, skipping sprouting.
Iter 1: no hypoxic zones, skipping sprouting.
Iter 2: no hypoxic zones, skipping sprouting.

Total angiogenic segments: 0



Inlet coords (6):
  0: [-9.742786 -5.625     6.1     ]
  1: [-1.76043e-15 -1.12500e+01  6.10000e+00]
  2: [ 9.742786 -5.625     6.1     ]
  3: [9.742786 5.625    6.1     ]
  4: [3.827021e-16 1.125000e+01 6.100000e+00]
  5: [-9.742786  5.625     6.1     ]
Central vein: [-6.328394e-08 -8.506539e-07 -6.000000e+00]
Arterial segs : 393
Venous segs   : 307
Capillary segs: 149
Angio segs    : 0
Hypoxic frac  : 0.00%
